# 🔧 Notebook 2 — Read-Repair: Heal Replicas as a Side Effect of Reads

**Read-repair** is a classic trick from Dynamo-style databases (Cassandra, DynamoDB, Riak):

> On every read, the coordinator asks **several** replicas for the same key. If it notices one of them returned a stale value, it **writes the fresh value back to the stale replica** — right there, during the read.

Convergence happens **lazily, paid for by reads**, instead of via a dedicated background scan.

In this notebook we build read-repair step by step (bad → better → best) and end with a side-by-side comparison vs. anti-entropy.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/read-repair
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🧱 A safer replica: last-write-wins on writes

Before building the coordinator, tighten the replica: a replica should **never go backwards**. If it already holds a newer value, ignore the older write. This is **last-write-wins (LWW)** applied on every write.


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Tuple, List, Optional
import random, time

Entry = Tuple[str, int]

@dataclass
class Replica:
    name: str
    data: Dict[str, Entry] = field(default_factory=dict)
    repairs_received: int = 0   # for stats

    def write(self, k: str, v: str, ts: int, *, is_repair: bool = False) -> bool:
        cur = self.data.get(k)
        if cur is None or ts > cur[1]:
            self.data[k] = (v, ts)
            if is_repair:
                self.repairs_received += 1
            return True
        return False

    def read(self, k: str) -> Optional[Entry]:
        return self.data.get(k)

def fresh_cluster():
    r1, r2, r3 = Replica("r1"), Replica("r2"), Replica("r3")
    r1.write("user:42", "Alice v2", 200)
    r2.write("user:42", "Alice v2", 200)
    r3.write("user:42", "Alice v1", 100)   # r3 is behind
    return [r1, r2, r3]


## 🟥 BAD: read-one, never repair (from Notebook 1)

We keep this as the baseline: stale reads, no healing.


In [ ]:
class ReadOneCoordinator:
    def __init__(self, replicas): self.replicas = replicas
    def read(self, k):
        r = random.choice(self.replicas)
        return r.name, r.read(k)

random.seed(0)
cluster = fresh_cluster()
co = ReadOneCoordinator(cluster)
for _ in range(5):
    print(co.read("user:42"))

print("\nr3 after 5 reads:", cluster[2].read("user:42"), "  <- still stale")
# Reads are pure: no amount of traffic heals anything.
assert cluster[2].read("user:42") == ("Alice v1", 100)


## 🟨 BETTER: quorum read — pick the freshest of R replicas

Ask `R` replicas, keep the one with the highest timestamp. This **fixes the client-visible staleness** but the stale replica is still stale on disk — next time we don't include it in the quorum, the same issue reappears.


In [ ]:
class QuorumReadCoordinator:
    def __init__(self, replicas): self.replicas = replicas

    def read(self, k, R=2, chosen=None):
        # Ask R replicas. In reality you would ask all N and take the first R replies;
        # which R you get is up to the network, so we let the caller pin them down.
        chosen = chosen if chosen is not None else self.replicas[:R]
        responses = [(r, r.read(k)) for r in chosen]
        valid = [(r, val) for r, val in responses if val is not None]
        if not valid: return None
        winner_r, (val, ts) = max(valid, key=lambda x: x[1][1])
        return val, ts

random.seed(0)
cluster = fresh_cluster()
co = QuorumReadCoordinator(cluster)
r1, r2, r3 = cluster

print("quorum(r1,r2):", co.read("user:42", chosen=[r1, r2]))
# The interesting quorum is the one that DOES include the stale replica. With R=2 and
# only one stale replica, any quorum still contains a fresh one, so the client is safe.
print("quorum(r2,r3):", co.read("user:42", chosen=[r2, r3]))
print("quorum(r1,r3):", co.read("user:42", chosen=[r1, r3]))
for pair in [(r1, r2), (r2, r3), (r1, r3)]:
    assert co.read("user:42", chosen=list(pair)) == ("Alice v2", 200), pair

print("\nr3 on disk :", cluster[2].read("user:42"), "  <- still stale!")
# The client is protected; the cluster is not. r3 has not been touched, so the moment
# a quorum forms without a fresh replica (or R drops to 1) the staleness resurfaces.
assert cluster[2].read("user:42") == ("Alice v1", 100)


## 🟩 BEST: read-repair — on every read, push the winner to stragglers

Do the quorum read, then for any replica whose value is older (or missing), **write the winner back** with the winning timestamp. Because replicas use LWW, repair writes are idempotent and safe to retry.


In [ ]:
class ReadRepairCoordinator:
    def __init__(self, replicas: List[Replica]):
        self.replicas = replicas
        self.repairs_triggered = 0

    def read(self, k: str, R: int = 2, chosen=None):
        chosen = chosen if chosen is not None else self.replicas[:R]
        responses = [(r, r.read(k)) for r in chosen]
        valid = [(r, val) for r, val in responses if val is not None]
        if not valid:
            return None
        _, (win_val, win_ts) = max(valid, key=lambda x: x[1][1])

        # Repair only the replicas we actually HEARD FROM. A coordinator has no idea
        # what a replica it never contacted is holding, and pushing a value at it would
        # be a blind write, not a repair. This is why read-repair converges the cluster
        # only as fast as reads happen to touch each replica — and why it needs
        # anti-entropy underneath it (notebook 3).
        for r, cur in responses:
            if cur is None or cur[1] < win_ts:
                print(f"  🔧 repairing {r.name}: {cur} -> ({win_val!r}, {win_ts})")
                r.write(k, win_val, win_ts, is_repair=True)
                self.repairs_triggered += 1
        return win_val, win_ts

random.seed(0)
cluster = fresh_cluster()
r1, r2, r3 = cluster
co = ReadRepairCoordinator(cluster)

# A quorum of (r1, r2) touches nothing stale — so it repairs nothing.
print("read (r1,r2):", co.read("user:42", chosen=[r1, r2]))
assert co.repairs_triggered == 0
assert r3.read("user:42") == ("Alice v1", 100), "r3 was never contacted, so it stays stale"
print("  -> r3 untouched, still", r3.read("user:42"), "(we never asked it anything)")

# A quorum that DOES include r3 heals it.
print("\nread (r2,r3):", co.read("user:42", chosen=[r2, r3]))
print("read (r2,r3):", co.read("user:42", chosen=[r2, r3]), "  <- nothing left to repair")
print("r3 on disk now:", r3.read("user:42"))
print("total repairs:", co.repairs_triggered)

assert r3.read("user:42") == ("Alice v2", 200), "the read should have healed r3"
assert co.repairs_triggered == 1, "and repaired exactly once — the second read is a no-op"
assert r3.repairs_received == 1
print("\n✔ read-repair heals a replica only when a read actually reaches it.")
print("  That is a feature (no blind writes) and a limitation (cold keys never heal).")


## ⏱️ Blocking vs. asynchronous repair

Above we repaired **inside** the client read path — the client waits for the repair write before seeing a response. That guarantees the stale replica is fixed *before* we return, but it raises tail latency.

A common optimization:

- **Blocking repair** — repair synchronously only when the freshest replica disagrees with the quorum majority. Used sparingly.
- **Asynchronous repair** — return the fresh value to the client immediately, then repair stragglers in a background task.

Below we simulate async repair with a simple queue.


In [ ]:
from collections import deque

class AsyncReadRepairCoordinator:
    def __init__(self, replicas: List[Replica]):
        self.replicas = replicas
        self.repair_queue: deque = deque()

    def read(self, k: str, R: int = 2, chosen=None):
        chosen = chosen if chosen is not None else self.replicas[:R]
        responses = [(r, r.read(k)) for r in chosen]
        valid = [(r, val) for r, val in responses if val is not None]
        if not valid: return None
        _, (win_val, win_ts) = max(valid, key=lambda x: x[1][1])
        for r, cur in responses:                 # again: only who we heard from
            if cur is None or cur[1] < win_ts:
                # Enqueue instead of writing now — client doesn't wait.
                self.repair_queue.append((r, k, win_val, win_ts))
        return win_val, win_ts      # returned immediately

    def drain_repairs(self):
        while self.repair_queue:
            r, k, v, ts = self.repair_queue.popleft()
            r.write(k, v, ts, is_repair=True)

cluster = fresh_cluster()
r1, r2, r3 = cluster
co = AsyncReadRepairCoordinator(cluster)
print("client sees:", co.read("user:42", chosen=[r2, r3]))
print("r3 BEFORE drain:", r3.read("user:42"))          # still stale momentarily
assert r3.read("user:42") == ("Alice v1", 100)
assert len(co.repair_queue) == 1, "the repair should be queued, not applied"

co.drain_repairs()                                     # the background worker runs
print("r3 AFTER drain :", r3.read("user:42"))
assert r3.read("user:42") == ("Alice v2", 200)
assert not co.repair_queue

# The window is real and it is the price of the lower latency: between the two
# asserts above, a second coordinator reading r3 alone would have seen "Alice v1".


**Trade-off:** async repair keeps read latency low, but there is a short window where a *different* coordinator reading the same key could still pick up a stale value. That's the "eventual" in eventual consistency.


## 🎲 Probabilistic read-repair (a real knob from Cassandra)

Cassandra used to expose `read_repair_chance` / `dclocal_read_repair_chance`: the probability, per read, that the coordinator queries **all** replicas (not just the quorum) and runs repair. With probability `1 - p` the read is cheap and skips repair.

Why? Reading all replicas on every request costs bandwidth. If your workload is read-heavy, most keys get repaired quickly even at `p = 0.1`.

Below we simulate the convergence rate for a few values of `p`.


In [ ]:
def simulate(p_repair: float, num_keys: int = 200, num_reads: int = 2000, seed: int = 0):
    random.seed(seed)
    # 3 replicas, r3 is missing every initial write
    reps = [Replica(f"r{i}") for i in range(3)]
    for k in range(num_keys):
        reps[0].write(f"k{k}", "v-new", 200)
        reps[1].write(f"k{k}", "v-new", 200)
        reps[2].write(f"k{k}", "v-old", 100)

    for _ in range(num_reads):
        k = f"k{random.randrange(num_keys)}"
        if random.random() < p_repair:
            # full repair path — ask everyone
            answers = [(r, r.read(k)) for r in reps]
            _, (val, ts) = max(answers, key=lambda x: x[1][1])
            for r in reps:
                cur = r.read(k)
                if cur is None or cur[1] < ts:
                    r.write(k, val, ts, is_repair=True)
        else:
            # cheap path — one replica, no repair
            random.choice(reps).read(k)

    still_stale = sum(1 for k in range(num_keys) if reps[2].data[f"k{k}"][1] < 200)
    return still_stale

results = {}
for p in (0.0, 0.05, 0.2, 0.5, 1.0):
    results[p] = simulate(p)
    print(f"p_repair={p:<4}  keys still stale on r3: {results[p]:>3}/200")

# p=0 must repair nothing at all — otherwise the comparison is meaningless.
assert results[0.0] == 200, results
# More repair probability must never leave MORE keys stale.
ps = sorted(results)
assert all(results[a] >= results[b] for a, b in zip(ps, ps[1:])), results
# And a modest 20% chance should already clear most of the backlog in 2000 reads.
assert results[0.2] < 0.25 * results[0.0], results
print(f"\n✔ p=0.2 cleared {100 * (1 - results[0.2]/200):.0f}% of the stale keys —"
      f" you do not need to repair on every read")


Notice how even small probabilities knock the stale count down fast — because each read is another chance to repair one of the 200 keys.


## 🧩 Gotcha: LWW is simple but lossy

Read-repair needs a rule to decide the "winner". LWW (highest timestamp) is the easiest, but if two clients write **concurrently**, LWW silently drops one of them. Production systems use richer schemes:

- **Vector clocks** (Riak, early Dynamo): detect concurrent writes and return *both* to the client as **siblings** — let the application merge.
- **CRDTs**: data types (counters, sets) whose merge function is built in and commutative.
- **Hybrid logical clocks (HLC)**: timestamps that are both wall-clock-ish and causality-aware.

For this lab we stay with LWW; just know it's not the end of the story. The `vector-clocks`
lab in this folder shows the detection half.

## 🧭 When you need read-repair — and when you don't

**Reach for it when:** replicas can diverge at all (any Dynamo-style quorum store), your reads
are spread across replicas, and the hot keys are the ones you care most about being fresh. It
is nearly free — you are already paying for the RPCs.

**Don't bother when:** every write goes through consensus (Raft/Paxos — etcd, Spanner,
CockroachDB), because replicas never disagree about committed data in the first place. Also
skip it when you only ever read from the leader: there is nothing to compare against.

**Never rely on it alone.** Read-repair fixes what gets read. Cold keys — the long tail, which
in most systems is the *majority* of keys — are never touched and stay broken forever. That is
what notebook 3 is about.
